# 🔬 BR-CP — Confirmatory study on a published TSFM (GPU, 11 November 2026)

**Purpose:** Run a **new, confirmatory frozen Chronos-Bolt study** on the BR-CP datasets.  Use the same GPU forecasts for SplitCP, RollingCP, ACI, AgACI, ConformalPID, BRCP, BRCP-safe and BRCP-data. Save recoverable checkpoints to Google Drive.

**Before running:** in Google Colab, select **Runtime → Change runtime type → GPU H100 / A100** (or L4), choose **High-RAM** if available. You must select GPU in the Colab interface yourself; this notebook cannot provision or purchase hardware.

**Data expected:** Existing Colab recovery folder: `MyDrive/BRCP_Recovery_2026-11-10/Class-1/` containing ETTh1, ETTh2, ETTm1, ETTm2, `exchange_rate.csv` and `national_illness.csv`.

**Scientific boundaries:** This is **not** a rerun of the historical global ridge forecaster; it is new evidence from a pretrained public checkpoint. A pretrained model's exposure to any evaluation datasets is not established as disjoint, so **do not claim strict zero-shot independence** until the model card/training corpus has been audited. It also does not automatically resolve all reviewer comments. All numerical results remain unverified until actual Colab output is reviewed. Unperturbed observational series are **not** guaranteed change-free nulls; empirical alarms are not exact false-positive rates.

**Recommended workflow:** run all cells in order. The first run can use `RUN_MODE='PILOT'` for a short compatibility test. For full confirmatory evidence change it to `RUN_MODE='FULL'` and rerun the configuration and execution cells. The GPU predictions are cached to Drive, and a disconnected run can be resumed using **Run all** with unchanged settings.

In [ ]:
# Check assigned accelerator *before* incurring model-download/compute costs.
import subprocess, sys
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True)
    print(result.stdout[:4000])
except Exception as e:
    raise RuntimeError('No NVIDIA GPU detected. Runtime > Change runtime type > select H100, A100, L4 or T4 GPU before continuing.') from e

In [ ]:
# Install needed inference library; Colab already supplies a GPU-capable PyTorch.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'chronos-forecasting', 'huggingface_hub', 'pandas', 'numpy', 'scipy', 'psutil'])
import torch, numpy as np, pandas as pd
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access CUDA; restart Colab with a GPU runtime.')
print('GPU:',torch.cuda.get_device_name(0), '| CUDA:',torch.version.cuda, '| PyTorch:',torch.__version__)
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory/(1024**3),1))

In [ ]:
# Mount your Google Drive (authorized by you only in the Colab interface).
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT=Path('/content/drive/MyDrive/BRCP_Recovery_2026-11-10/Class-1')
OUT_ROOT=Path('/content/drive/MyDrive/BRCP_Chronos_Bolt_GPU_Confirm_2026-11-11')
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Expected recovered BRCP data folder not found: {DATA_ROOT}')
OUT_ROOT.mkdir(parents=True,exist_ok=True)
print('Input:',DATA_ROOT)
print('Saved results:',OUT_ROOT)

In [ ]:
# Embedded, audited BR-CP source + GPU runner. No separate ZIP download needed.
import base64, io, zipfile, pathlib, sys
PAYLOAD_B64 = "..."
code_dir=pathlib.Path('/content/brcp_gpu_confirm_code')
code_dir.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD_B64))) as z:
    z.extractall(code_dir)
sys.path.insert(0,str(code_dir))
import hashlib
print('Extracted',len(list(code_dir.rglob('*.py'))),'Python source files')
print('Runner SHA-256:',hashlib.sha256((code_dir/'runner.py').read_bytes()).hexdigest())
import runner
print('Available methods:',', '.join(runner.BASE_METHODS))

## Choose study size and GPU model

For **H100/A100**, the notebook defaults to **Chronos-Bolt Base** (larger published checkpoint); for **L4/T4**, it selects **Chronos-Bolt Small** to reduce memory pressure and runtime. Both are frozen pretrained models. Model selection, batch size, dataset filenames, seeds and every result are recorded.

- `FULL`: six datasets, up to seven eligible columns each, 12,000 observations per series, unperturbed + synthetic level/variance interventions. It may consume substantial Colab compute units. Progress checkpoints survive disconnects.
- `PILOT`: same protocols at smaller scale to confirm the environment, **not** a complete confirmatory paper result.

The first eligible column of each dataset is *development only*. Evaluation uses other columns, preventing direct threshold selection on test series. This doesn't ensure independent sequences statistically.

In [ ]:
import torch
RUN_MODE='FULL'   # Change to 'PILOT' for a short trial first.
gpu=torch.cuda.get_device_name(0).upper()
MODEL_NAME='amazon/chronos-bolt-base' if any(x in gpu for x in ('H100','A100')) else 'amazon/chronos-bolt-small'
# You may override MODEL_NAME above if your available GPU memory is sufficient.
CONFIG={
 'data_root':str(DATA_ROOT), 'output_root':str(OUT_ROOT),
 'datasets':['ETTh1','ETTh2','ETTm1','ETTm2','exchange','illness'],
 'model':MODEL_NAME, 'seed':20261111, 'context':256,
 'alpha':0.10, 'warm':300, 'window':300, 'pre':100,
 'horizon_cd':150, 'segment':120, 'gap':1000, 'magnitude':3.0,
 'k':0.80, 'kinds':['none','level','variance'],
 'max_cols_per_dataset':7 if RUN_MODE=='FULL' else 3,
 'max_rows':12000 if RUN_MODE=='FULL' else 2000,
 'batch':96 if any(x in gpu for x in ('H100','A100')) else 24,
 'h_candidates':[4,6,8,10,12,16,20],
 'target_alarms_per800':0.50,
 'test_rate_tolerance_per800':0.15,
}
print('RUN_MODE:', RUN_MODE, 'MODEL:', MODEL_NAME, 'BATCH:', CONFIG['batch'])
print('No model fine-tuning. Fitted/calibration data are restricted to pre-outcome information.')

In [ ]:
# Execute new confirmatory study. Multiple interrupted runs can resume from Drive.
# This line may take a while. Watch for [CACHE], [RESUME] and [DONE].
from runner import run
run_dir = run(CONFIG)
print('FINISHED. Evidence directory:',run_dir)

In [ ]:
# Export a compact, shareable scientific audit ZIP (the full forecast caches remain in Drive).
import zipfile, json, os, sys, hashlib, shutil
from pathlib import Path
run_dir=Path(run_dir)
file_list=['RUN_MANIFEST.json','dev_detector_tuning.csv','summary_by_kind_and_method.csv',
           'all_method_results.csv','event_level_results.csv','paired_event_comparison.csv',
           'exploratory_paired_bootstrap.csv']
archive=run_dir.parent/('BRCP_Chronos_GPU_Evidence_'+run_dir.name+'.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for filename in file_list:
        p=run_dir/filename
        if p.exists(): z.write(p,arcname=filename)
    for src in sorted(Path('/content/brcp_gpu_confirm_code').rglob('*.py')):
        z.write(src,arcname='source/'+str(src.relative_to('/content/brcp_gpu_confirm_code')))
print('Evidence ZIP:',archive)
print('Archive SHA-256:', hashlib.sha256(archive.read_bytes()).hexdigest())
print('SEND THIS ZIP BACK TO CHATGPT for scientific verification before changing the manuscript.')

### Interpretation checklist (important)

1. **Do not label a GPU job successful until `RUN_MANIFEST.json` says `stage: completed`.**
2. Inspect `test_detector_rates.status`. If `UNMATCHED`, the detector comparison remains inconclusive even after selecting thresholds on development data.
3. `ConformalPID` in this repository is a **simplified PI instantiation**. `AgACI` is an implemented aggregation variant, not an independently validated full published BOA-AgACI benchmark. The manuscript must be precise.
4. The checkpoint may have trained on some benchmark series; do **not** claim disjoint zero-shot validation without a training-corpus audit.
5. Differences in synthetic intervention protocol from the historical run mean the results are **additional** evidence, not a replacement for the earlier reported tables. Do not mix their numbers into the main table without reworking the experimental design.
6. **No result is prefilled**. Every reported value must come from the actual saved CSV/NPZ files, with provenance and exact source SHA-256 hashes.
